# Presença e sessões da frota (Gold)

Resume os snapshots de posição em janelas de presença por motorista. Serve ao Python para: (a) janelas `available_from/available_until` dos veículos no replay; (b) histórico de presença por dia da semana e hora para estimar a frota de D+1; (c) posição conhecida no início de cada hora.

## Fluxo dos dados

* Origem: `cat_mobilidade_urbana_silver.db_frota.tb_posicoes_motoristas`
* Destinos em `cat_mobilidade_urbana_gold.db_frota`: `tb_frota_sessoes`, `tb_frota_motorista_hora`

## Regras

* **Sessão**: sequência de snapshots consecutivos do mesmo motorista sem intervalo maior que 300 s e sem troca entre disponível (`ONLINE`, `SELECIONADO`, `A_CAMINHO`, `COM_PASSAGEIRO`) e `INDISPONIVEL`. Ausência nos snapshots significa offline.
* A duração de cada snapshot vai até o snapshot seguinte da mesma sessão; o último snapshot da sessão tem duração zero, então os minutos são um limite inferior.
* `ONLINE` indica aptidão para receber corrida, confirmada pelo proponente; não prova veículo livre de compromissos externos (RN015, RN026).
* O motorista é identificado por pseudônimo determinístico; o código bruto fica apenas na Silver.

In [0]:
# No Databricks `spark` já existe; no VS Code o Databricks Connect cria uma sessão serverless.
try:
    spark  # noqa: F821
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless(True).getOrCreate()

from pyspark.sql import Window as W
from pyspark.sql import functions as F

# Todos os carimbos da fonte têm offset -03:00; datas e horas locais usam o fuso informado pelo proponente.
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

BRONZE = "cat_mobilidade_urbana_bronze.db_frota"
SILVER = "cat_mobilidade_urbana_silver.db_frota"
GOLD = "cat_mobilidade_urbana_gold.db_frota"


def segundos_entre(inicio, fim):
    """Diferença em segundos (com fração) entre dois timestamps."""
    return F.col(fim).cast("double") - F.col(inicio).cast("double")


def haversine_km(lat1, lon1, lat2, lon2):
    """Mesmo Haversine de src/tcc_frota/distance.py (R = 6371,0088 km).

    Linha reta entre dois pontos: estimativa geodésica, nunca rodagem observada (RN012).
    """
    la1, lo1, la2, lo2 = (F.radians(F.col(c)) for c in (lat1, lon1, lat2, lon2))
    h = F.pow(F.sin((la2 - la1) / 2), 2) + F.cos(la1) * F.cos(la2) * F.pow(F.sin((lo2 - lo1) / 2), 2)
    return F.lit(6371.0088) * 2 * F.asin(F.least(F.lit(1.0), F.sqrt(h)))


def pseudonimo(coluna):
    """Pseudônimo determinístico (SHA-256 truncado) para tabelas Gold; não é anonimização."""
    return F.substring(F.sha2(F.concat(F.lit("tcc-frota|"), F.col(coluna).cast("string")), 256), 1, 16)


def gravar(df, tabela, comentario):
    """Sobrescreve a tabela Delta de forma idempotente e registra o comentário."""
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    print(f"{tabela}: {spark.table(tabela).count():,} linhas")

In [0]:
p = (
    spark.table(f"{SILVER}.tb_posicoes_motoristas")
    .select(
        "id_motorista", "ts_snapshot", "estado", "latitude", "longitude", "h3_r8",
        (F.col("fl_coordenada_valida") & F.col("fl_posicao_atual")).alias("fl_posicao_ok"),
    )
    .withColumn("fl_disponivel", F.col("estado") != "INDISPONIVEL")
)
o = W.partitionBy("id_motorista").orderBy("ts_snapshot")
p = (
    p.withColumn("_ts_ant", F.lag("ts_snapshot").over(o))
    .withColumn("_disp_ant", F.lag("fl_disponivel").over(o))
    .withColumn(
        "_nova",
        (
            F.col("_ts_ant").isNull()
            | (segundos_entre("_ts_ant", "ts_snapshot") > 300)
            | (F.col("fl_disponivel") != F.col("_disp_ant"))
        ).cast("int"),
    )
    .withColumn("seq_sessao", F.sum("_nova").over(o.rowsBetween(W.unboundedPreceding, W.currentRow)))
)
os_ = W.partitionBy("id_motorista", "seq_sessao").orderBy("ts_snapshot")
p = (
    p.withColumn("_prox", F.lead("ts_snapshot").over(os_))
    .withColumn("seg_duracao", F.coalesce(segundos_entre("ts_snapshot", "_prox"), F.lit(0.0)))
)


def minutos(estado):
    return F.round(F.sum(F.when(F.col("estado") == estado, F.col("seg_duracao")).otherwise(0)) / 60, 2)


def posicao(funcao, sufixo):
    ordem = F.when(F.col("fl_posicao_ok"), F.col("ts_snapshot"))
    return [
        funcao(F.col("latitude"), ordem).alias(f"lat_{sufixo}"),
        funcao(F.col("longitude"), ordem).alias(f"lon_{sufixo}"),
        funcao(F.col("h3_r8"), ordem).alias(f"h3_r8_{sufixo}"),
    ]


sessoes = (
    p.groupBy("id_motorista", "seq_sessao")
    .agg(
        F.min("ts_snapshot").alias("ts_inicio"),
        F.max("ts_snapshot").alias("ts_fim"),
        F.first("fl_disponivel").alias("fl_disponivel"),
        F.count("*").alias("qtd_snapshots"),
        minutos("ONLINE").alias("min_online"),
        minutos("SELECIONADO").alias("min_selecionado"),
        minutos("A_CAMINHO").alias("min_a_caminho"),
        minutos("COM_PASSAGEIRO").alias("min_com_passageiro"),
        minutos("INDISPONIVEL").alias("min_indisponivel"),
        *posicao(F.min_by, "inicio"),
        *posicao(F.max_by, "fim"),
    )
    .withColumn("estado_sessao", F.when(F.col("fl_disponivel"), "DISPONIVEL").otherwise("INDISPONIVEL"))
    .withColumn("min_duracao", F.round(segundos_entre("ts_inicio", "ts_fim") / 60, 2))
    .withColumn("data_local_inicio", F.to_date("ts_inicio"))
    .withColumn("id_motorista_pseud", pseudonimo("id_motorista"))
    .drop("id_motorista", "fl_disponivel")
)
gravar(
    sessoes,
    f"{GOLD}.tb_frota_sessoes",
    "Sessoes de presenca por motorista (pseudonimo): snapshots consecutivos ate 300 s sem trocar entre disponivel e "
    "INDISPONIVEL. Minutos por estado e posicoes inicial e final.",
)

In [0]:
por_hora = (
    p.withColumn("data_local", F.to_date("ts_snapshot"))
    .withColumn("hora_local", F.hour("ts_snapshot"))
    .groupBy("id_motorista", "data_local", "hora_local")
    .agg(
        F.count("*").alias("qtd_snapshots"),
        F.max("fl_disponivel").alias("fl_disponivel_na_hora"),
        minutos("ONLINE").alias("min_online"),
        minutos("SELECIONADO").alias("min_selecionado"),
        minutos("A_CAMINHO").alias("min_a_caminho"),
        minutos("COM_PASSAGEIRO").alias("min_com_passageiro"),
        minutos("INDISPONIVEL").alias("min_indisponivel"),
        *posicao(F.min_by, "primeira"),
    )
    .withColumn("dia_semana_iso", ((F.dayofweek("data_local") + 5) % 7) + 1)
    .withColumn("id_motorista_pseud", pseudonimo("id_motorista"))
    .drop("id_motorista")
)
gravar(
    por_hora,
    f"{GOLD}.tb_frota_motorista_hora",
    "Presenca de cada motorista (pseudonimo) por data e hora local: minutos por estado e primeira posicao valida da hora.",
)